# 04 Delta Expert Optimizer

This loop goes beyond convex `G/A/S` mixing.  It uses `G` as the parent anchor
and learns how much of the aggregate delta and repair delta to inject:

```text
M = G + alpha(A - G) + beta(S - G)
```

The coefficients are independent for `body`, `head`, `router`, and each MoE
expert.  The purpose is to find a judge that can keep improving across rounds
instead of merely freezing when repeated repair drifts.


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path('/app/Object_Detection')
PROJECT = ROOT / 'dynamic_quality_aware_classwise_aggregation' / 'moe_dqa_judger'
OUT = PROJECT / 'output' / '04_delta_expert_optimizer'
OUT


PosixPath('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/04_delta_expert_optimizer')

## Run Loop


In [2]:
import subprocess, sys

cmd = [
    sys.executable,
    str(PROJECT / 'scripts' / 'run_04_delta_expert_optimizer.py'),
    '--workspace-root', str(OUT),
    '--rounds', '1,2,3,4,5,6',
    '--mini-images', '512',
    '--random-candidates', '10',
    '--surrogate-iterations', '1',
    '--surrogate-pool', '72',
    '--surrogate-evals', '3',
    '--full-eval-topk', '2',
    '--val-batch-size', '32',
    '--notify-discord',
    '--force',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_04_delta_expert_optimizer.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/04_delta_expert_optimizer --rounds 1,2,3,4,5,6 --mini-images 512 --random-candidates 10 --surrogate-iterations 1 --surrogate-pool 72 --surrogate-evals 3 --full-eval-topk 2 --val-batch-size 32 --notify-discord --force


[
  {
    "round": 2.0,
    "candidate_id": "best00_sur00_01",
    "phase": "full",
    "eval_scope": "full_total",
    "path": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/04_delta_expert_optimizer/candidates/r002_best00_sur00_01.pt",
    "g_map50": 0.52697,
    "a_proxy_map50": 0.522605,
    "s_map50": 0.52276,
    "pseudo_mean_score": 0.822722258171578,
    "pseudo_mean_conf": 0.8445061213919342,
    "pseudo_mean_stability": 0.9714267354923062,
    "pseudo_boxes": 9943.0,
    "pseudo_images": 2528.0,
    "class_entropy_norm": 0.4671684933410906,
    "rare_fraction": 0.0066378356632807,
    "vehicle_fraction": 0.9182339334204969,
    "expert_entropy_norm": 0.7764041285745504,
    "dead_expert_fraction": 0.25,
    "repair_gain_vs_a": 0.00015500000000001624,
    "repair_gain_vs_g": -0.004210000000000047,
    "body_a": 0.02116942695271451,
    "body_s": -0.08835673527944735,
    "head_a": 0.10696586272792306,
    "head_s": 0.01419381693852384,

CompletedProcess(args=['/opt/venv/bin/python3', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_04_delta_expert_optimizer.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/04_delta_expert_optimizer', '--rounds', '1,2,3,4,5,6', '--mini-images', '512', '--random-candidates', '10', '--surrogate-iterations', '1', '--surrogate-pool', '72', '--surrogate-evals', '3', '--full-eval-topk', '2', '--val-batch-size', '32', '--notify-discord', '--force'], returncode=0)

## Results


In [3]:
best = pd.read_csv(OUT / 'stats' / '04_delta_expert_best_full.csv')
cols = [
    'round','candidate_id','map50','map50_95','precision','recall','score',
    'body_a','body_s','head_a','head_s','router_a','router_s',
    'expert0_a','expert0_s','expert1_a','expert1_s','expert2_a','expert2_s','expert3_a','expert3_s'
]
display(best[cols].head(20))
print((OUT / '04_delta_expert_optimizer_report.md').read_text())


,round,candidate_id,map50,map50_95,precision,recall,score,body_a,body_s,head_a,...,router_a,router_s,expert0_a,expert0_s,expert1_a,expert1_s,expert2_a,expert2_s,expert3_a,expert3_s
0,2.0,best00_sur00_01,0.462,0.260,0.693,0.431,0.57455,0.021169,-0.088357,0.106966,...,0.125791,-0.042307,0.197781,-0.147791,0.178042,0.094847,0.314919,0.113174,0.338327,-0.003278
1,2.0,best01_sur00_00,0.462,0.260,0.697,0.430,0.57450,0.395491,-0.131763,0.087545,...,0.380497,0.157212,0.339631,-0.086181,0.314441,0.099447,0.282363,0.147832,0.460047,-0.095562
2,1.0,best00_sur00_00,0.462,0.258,0.714,0.424,0.57350,-0.236697,0.286205,0.330059,...,0.516741,0.241045,0.582112,0.356368,0.509721,-0.250000,0.414197,0.263688,0.220533,0.637585
3,1.0,best01_rand007,0.462,0.258,0.715,0.424,0.57350,-0.168320,0.235776,0.369925,...,0.651977,0.262847,0.537384,0.191399,0.505010,-0.201877,0.339853,0.075145,0.086289,0.432957
4,3.0,best00_sur00_02,0.459,0.258,0.717,0.418,0.57020,0.494323,-0.250000,0.440574,...,0.172249,-0.147826,0.965843,0.183158,0.465279,0.247791,1.250000,0.107022,0.776733,0.463894
5,3.0,best01_sur00_00,0.459,0.258,0.719,0.417,0.57015,0.646447,-0.250000,0.365673,...,0.126143,-0.250000,0.975996,-0.027394,0.766722,0.195454,1.173213,0.099488,0.707884,0.337585
6,4.0,best01_rand000,0.456,0.256,0.690,0.430,0.56710,-0.250000,-0.250000,-0.250000,...,0.461822,-0.161949,0.600290,0.556600,0.014471,-0.072641,-0.250000,-0.231911,0.081304,0.003409
7,4.0,best00_rand008,0.456,0.256,0.693,0.428,0.56700,-0.250000,-0.141565,-0.249911,...,-0.127745,-0.200323,-0.086122,-0.067146,0.218406,-0.159519,0.193466,-0.250000,0.097136,0.140448
8,5.0,best01_sur00_02,0.450,0.253,0.689,0.423,0.55970,0.946421,-0.082509,0.282111,...,1.250000,-0.250000,1.061950,-0.035867,1.020578,-0.159326,0.940026,-0.075398,0.893685,-0.016110
9,5.0,best00_slight_extrapolate_target,0.450,0.253,0.689,0.422,0.55965,1.050000,-0.050000,0.300000,...,1.100000,-0.050000,1.100000,-0.050000,1.100000,-0.050000,0.900000,0.050000,0.900000,0.050000


# DQA-SoftMoX Delta Expert Optimizer 04

- created_utc: 2026-05-13T14:00:11.361606+00:00
- mini_images: 512

| rank | round | candidate | mAP50 | mAP50:95 | score | body a/s | head a/s | router a/s | expert0 a/s | expert1 a/s | expert2 a/s | expert3 a/s |
|---:|---:|---|---:|---:|---:|---|---|---|---|---|---|---|
| 1 | 2.0 | best00_sur00_01 | 0.462 | 0.260 | 0.5746 | 0.02/-0.09 | 0.11/0.01 | 0.13/-0.04 | 0.20/-0.15 | 0.18/0.09 | 0.31/0.11 | 0.34/-0.00 |
| 2 | 2.0 | best01_sur00_00 | 0.462 | 0.260 | 0.5745 | 0.40/-0.13 | 0.09/0.38 | 0.38/0.16 | 0.34/-0.09 | 0.31/0.10 | 0.28/0.15 | 0.46/-0.10 |
| 3 | 1.0 | best00_sur00_00 | 0.462 | 0.258 | 0.5735 | -0.24/0.29 | 0.33/0.79 | 0.52/0.24 | 0.58/0.36 | 0.51/-0.25 | 0.41/0.26 | 0.22/0.64 |
| 4 | 1.0 | best01_rand007 | 0.462 | 0.258 | 0.5735 | -0.17/0.24 | 0.37/0.67 | 0.65/0.26 | 0.54/0.19 | 0.51/-0.20 | 0.34/0.08 | 0.09/0.43 |
| 5 | 3.0 | best00_sur00_02 | 0.459 | 0.258 | 0.5702 | 0.49/-0.25 | 0.44/0.54 | 0.17/-0.15 | 0.97/0.18 | 0.47/0.25 | 1.